# EEA exploration and source choice

This notebook records my first exploration of Paris PM2.5 data from the European Environment Agency (EEA). I started with the EEA because it provides one interface for several European countries, which is useful for the comparator analysis.

The main result is that FR04329 is the only Paris station whose PM2.5 observations span the full 2013-2024 period in this extract. The number of available stations changes over time, so a simple average across whichever stations are present could partly reflect changes in the monitoring network. The later panel analysis therefore includes station fixed effects, so permanent differences between stations are not treated as changes over time.

I later rebuilt the French series using Geod'Air, the French national air-quality database operated through INERIS/LCSQA. It does not change which physical stations existed, but it provides the French source used in the main analysis through 2025. The EEA data remain useful for European comparators and for historical NO₂ before 2013. This notebook records how the source choice was made.

## Data and references

**Data**
- [EEA Air Quality Download Service](https://www.eea.europa.eu/en/datahub/datahubitem-view/778ef9f5-6293-4846-badd-56a29c70880d): station metadata and verified hourly observations reported by participating countries.
- [Geod'Air](https://www.geodair.fr/) (INERIS/LCSQA): French data source used from notebook 02 onward.

**Data conventions**
- In the [EEA observation-validity vocabulary](https://dd.eionet.europa.eu/vocabulary/aq/observationvalidity/), code 1 means valid. Only those observations are used here.
- Negative readings that EEA marks as valid are retained rather than set to zero or deleted. Jiang et al. (2023) explain that negative PM2.5 readings can arise from measurement uncertainty and that the chosen threshold affects reported statistics: Jiang, N. et al. (2023), *On thresholds for controlling negative particle (PM2.5) readings in air quality reporting*, Environmental Monitoring and Assessment, 195(10), 1187. [doi:10.1007/s10661-023-11750-4](https://doi.org/10.1007/s10661-023-11750-4).

**Software**
- pandas: McKinney, W. (2010). Data structures for statistical computing in Python. Proceedings of the 9th Python in Science Conference, 56-61.
- [`airbase`](https://airbase.readthedocs.io/en/stable/) is an independent Python client for the EEA download service; its source code is available on [GitHub](https://github.com/JohnPaton/airbase).


## 1. Load the EEA station metadata

The `airbase` downloader uses asynchronous requests. Applying `nest_asyncio` once allows those requests to run inside Jupyter's existing event loop. I then download the station metadata.

In [1]:
import glob

import airbase
import nest_asyncio
import pandas as pd

nest_asyncio.apply()

In [2]:
client = airbase.AirbaseClient()
client.download_metadata("data/meta/stations.csv")

Writing metadata to data/meta/stations.csv...


### Inspect the metadata

I first check the size and column names before applying any filters.

In [3]:
stations = pd.read_csv("data/meta/stations.csv", low_memory=False)
print(stations.shape)
print(stations.columns.tolist())

(187198, 70)
['Country', 'B-G Namespace', 'Year', 'Air Quality Network', 'Air Quality Network Name', 'Timezone', 'Air Quality Station EoI Code', 'Air Quality Station Nat Code', 'Air Quality Station Name', 'Sampling Point Id', 'Air Pollutant', 'Longitude', 'Latitude', 'Altitude', 'Altitude Unit', 'Air Quality Station Area', 'Air Quality Station Type', 'Operational Activity Begin', 'Operational Activity End', 'Sample Id', 'Inlet Height', 'Inlet Height Unit', 'Building Distance', 'Building Distance Unit', 'Kerb Distance', 'Kerb Distance Unit', 'Distance Source', 'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions', 'Heating Emissions Unit', 'Mobile', 'Traffic Emissions', 'Traffic Emissions Unit', 'Industrial Emissions', 'Industrial Emissions Unit', 'Municipality', 'Dispersion Local', 'Dispersion Regional', 'Distance Junction', 'Distance Junction Unit', 'Heavy Duty Fraction', 'Height Facades', 'Street Width', 'Traffic Speed', 'Traffic Volume', 'Process Id', 'Process Activit

### Select French PM2.5 stations

I filter the metadata to France and PM2.5, then inspect the municipality names before defining the Paris filter.

In [4]:
fr_pm25 = stations[
    (stations["Country"] == "France")
    & (stations["Air Pollutant"] == "PM2.5")
]
print(fr_pm25.shape)
print(fr_pm25["Municipality"].dropna().unique()[:50])

(906, 70)
<ArrowStringArray>
[           'GENNEVILLIERS', 'PARIS 18E ARRONDISSEMENT',
                  'GONESSE',  'PARIS 8E ARRONDISSEMENT',
          'VITRY-SUR-SEINE',   'SAINT-MARTIN-DU-TERTRE',
 'PARIS 1ER ARRONDISSEMENT',              'SAINT-DENIS',
              'BOIS-HERPIN',                    'MELUN',
  'PARIS 9E ARRONDISSEMENT',  'PARIS 4E ARRONDISSEMENT',
                  'PUTEAUX',                  'BOBIGNY',
              'COULOMMIERS',                'MONTLHÉRY',
              'RAMBOUILLET',            'FONTAINEBLEAU',
 'PARIS 12E ARRONDISSEMENT',                   'DONGES',
                  'FROSSAY', 'SAINT-ÉTIENNE-DE-MONTLUC',
                   'NANTES',            'SAINT-NAZAIRE',
                   'ANGERS',                    'LAVAL',
              'LA TARDIÈRE',         'LA ROCHE-SUR-YON',
      'SAINT-DENIS-D'ANJOU',                  'LE MANS',
                     'REZÉ',               'BOUGUENAIS',
      'MONTOIR-DE-BRETAGNE',              'LA ROCHELLE',
  

### Select stations inside Paris

Paris municipalities are recorded by arrondissement. Matching names that start with `PARIS ` retains stations inside the city and excludes surrounding municipalities. I keep both traffic and background stations.

In [5]:
paris_meta = fr_pm25[
    fr_pm25["Municipality"].str.startswith("PARIS ", na=False)
]
station_columns = [
    "Air Quality Station EoI Code",
    "Air Quality Station Name",
    "Air Quality Station Type",
    "Air Quality Station Area",
    "Municipality",
    "Air Quality Network Name",
    "Year",
]
print(
    paris_meta[station_columns]
    .drop_duplicates()
    .sort_values("Air Quality Station Name")
)

      Air Quality Station EoI Code Air Quality Station Name  \
88408                      FR04031        Av Champs Elysees   
89203                      FR04329     Bld peripherique Est   
88876                      FR04131      Boulevard Haussmann   
88198                      FR04004              PARIS 18eme   
88607                      FR04055     PARIS 1er Les Halles   
88939                      FR04143             PARIS Centre   

      Air Quality Station Type Air Quality Station Area  \
88408                  traffic                    urban   
89203                  traffic                    urban   
88876                  traffic                    urban   
88198               background                    urban   
88607               background                    urban   
88939               background                    urban   

                   Municipality Air Quality Network Name    Year  
88408   PARIS 8E ARRONDISSEMENT                 AIRPARIF  2025.0  
89203  PAR

### Save the station list

The six EoI station codes are saved for later checks and used below to select their measurement records.

In [6]:
paris_codes = sorted(paris_meta["Air Quality Station EoI Code"].unique())
print(paris_codes)
paris_meta[station_columns].drop_duplicates().to_csv(
    "data/meta/paris_stations.csv", index=False
)

['FR04004', 'FR04031', 'FR04055', 'FR04131', 'FR04143', 'FR04329']


## 2. Load the hourly PM2.5 measurements

The `Verified` request selects the E1a series. The `airbase` documentation describes this release as verified observations for 2013-2024, while 2025 observations are supplied separately as unverified data.

The download is large, so I reuse Parquet files already under `data/raw`. This folder should contain the French PM2.5 download rather than unrelated EEA files. The sampling-point and series checks below stop the notebook if other pollutants or aggregation types have been mixed into the selected Paris records.

In [7]:
files = glob.glob("data/raw/**/*.parquet", recursive=True)

if not files:
    request = client.request("Verified", "FR", poll="PM2.5")
    request.download(dir="data/raw", skip_existing=True)
    files = glob.glob("data/raw/**/*.parquet", recursive=True)

assert files, "No Parquet files were found in data/raw"
print("Parquet files found:", len(files))

Parquet files found: 313


### Combine the measurement files

The download contains separate Parquet files for different stations and years. I combine them and inspect the resulting columns before selecting Paris.

In [8]:
measurements = pd.concat(
    (pd.read_parquet(file) for file in files),
    ignore_index=True,
)
print(measurements.columns.tolist())
print(len(measurements))

['Samplingpoint', 'Pollutant', 'Start', 'End', 'Value', 'Unit', 'AggType', 'Validity', 'Verification', 'ResultTime', 'DataCapture', 'FkObservationLog']
17995236


### Match the measurement and station identifiers

The hourly files use sampling-point identifiers such as `FR/SPO-FR04329_6001`, whereas the metadata use EoI station codes such as `FR04329`. The identifiers contain the station code and end with the EEA pollutant code. [Code 6001 is PM2.5](https://dd.eionet.europa.eu/vocabulary/aq/pollutant/6001), so I use both parts as a check after selecting the six Paris stations. I also require a single pollutant, unit and aggregation type before continuing.

In [9]:
station_pattern = "|".join(paris_codes)
paris_hourly = measurements[
    measurements["Samplingpoint"].str.contains(station_pattern, na=False)
].copy()

assert paris_hourly["Samplingpoint"].str.endswith("_6001").all(), (
    "The selected records include a pollutant other than PM2.5"
)
series_fields = ["Pollutant", "Unit", "AggType"]
assert len(paris_hourly[series_fields].drop_duplicates()) == 1, (
    "More than one pollutant, unit or aggregation type was selected"
)

print(len(paris_hourly))
print(paris_hourly["Samplingpoint"].unique())

278283
<ArrowStringArray>
['FR/SPO-FR04031_6001', 'FR/SPO-FR04055_6001', 'FR/SPO-FR04131_6001',
 'FR/SPO-FR04329_6001', 'FR/SPO-FR04004_6001', 'FR/SPO-FR04143_6001']
Length: 6, dtype: str


### Clean the measurements

I convert `Value` to numeric form and retain only EEA validity code 1, which the data dictionary defines as valid. Rows that cannot be converted are removed.

I do not apply a second filter based on the sign of the concentration. The smallest retained value is only -3.0 µg/m³, and EEA has already marked these records as valid. Deleting or replacing them with zero would truncate the lower tail. I print their number so that this choice remains visible.

In [10]:
paris_hourly["Value"] = pd.to_numeric(
    paris_hourly["Value"], errors="coerce"
)
paris_hourly = paris_hourly[paris_hourly["Validity"] == 1].copy()
paris_hourly = paris_hourly.dropna(subset=["Value"])

negative_count = (paris_hourly["Value"] < 0).sum()
print(len(paris_hourly), "valid rows;", negative_count,
      "negative readings retained")
print(paris_hourly["Value"].describe())

267623 valid rows; 534 negative readings retained
count    267623.000000
mean         13.607436
std          10.522743
min          -3.000000
25%           6.900000
50%          10.600000
75%          17.100000
max         169.980000
Name: Value, dtype: float64


## 3. Aggregate to daily station means

I parse the timestamps and check that none failed before calculating a daily mean for each sampling point. This table is descriptive and is not used to impose a daily completeness threshold.

Pandas inserts calendar-day rows between the first and last observation of each station when resampling, including `NaN` on days with no valid hourly value. The year counts below therefore describe the resampled table rather than proving complete daily coverage. The verified EEA release is used for 2013-2024; the five-row 2025 tail is not used. The main Geod'Air analysis supplies the 2025 French data.

In [11]:
paris_hourly["Start"] = pd.to_datetime(
    paris_hourly["Start"], errors="coerce"
)
assert paris_hourly["Start"].notna().all(), "Unparsed timestamps found"

print("EEA extract spans:", paris_hourly["Start"].dt.year.min(),
      "to", paris_hourly["Start"].dt.year.max())
print("negative valid readings in the EEA extract:", negative_count)

paris_daily = (paris_hourly
         .set_index("Start")
         .groupby("Samplingpoint")["Value"]
         .resample("D")
         .mean()
         .reset_index())
print(paris_daily.shape)
print(paris_daily["Start"].dt.year.value_counts().sort_index())

EEA extract spans: 2013 to 2025
negative valid readings in the EEA extract: 534
(11691, 3)
Start
2013     730
2014     730
2015     730
2016     732
2017     730
2018     730
2019     724
2020     732
2021     730
2022    1463
2023    1825
2024    1830
2025       5
Name: count, dtype: int64


## 4. Check changes in station coverage

The number of available station records is lower before 2022. I compare the operational dates in the metadata with the first and last observations in the measurement files.

The operational start date is not always the same as the start of the selected PM2.5 sampling series. For example, FR04055 was commissioned in 2002, but its PM2.5 records in this extract start in 2019. FR04143 ends in September 2019, three stations start reporting PM2.5 during 2022, and FR04329 is the only station whose first and last observations span 2013-2024. This is a span check, not a claim that every hour is present.

This matters because stations that enter later may have different average pollution levels. A citywide mean calculated from whichever stations are available could therefore mix pollution change with station composition. In the later panel, station fixed effects absorb stable level differences between stations; they do not create observations for periods in which a station is absent. Switching from EEA to Geod'Air does not change the physical station coverage.

In [12]:
date_columns = [
    "Air Quality Station EoI Code",
    "Air Quality Station Name",
    "Operational Activity Begin",
    "Operational Activity End",
]
station_dates = (
    paris_meta[date_columns]
    .drop_duplicates()
    .sort_values("Operational Activity Begin")
)
print(station_dates)

      Air Quality Station EoI Code Air Quality Station Name  \
88198                      FR04004              PARIS 18eme   
88876                      FR04131      Boulevard Haussmann   
88607                      FR04055     PARIS 1er Les Halles   
88939                      FR04143             PARIS Centre   
88408                      FR04031        Av Champs Elysees   
89203                      FR04329     Bld peripherique Est   

      Operational Activity Begin Operational Activity End  
88198        01/01/2022 00:00:00                      NaN  
88876        01/01/2022 00:00:00                      NaN  
88607        03/01/2002 00:00:00                      NaN  
88939        16/12/2011 00:00:00      24/09/2019 00:00:00  
88408        23/12/2022 00:00:00                      NaN  
89203        25/12/2012 00:00:00                      NaN  


In [13]:
coverage = (
    paris_hourly.groupby("Samplingpoint")["Start"]
    .agg(["min", "max", "count"])
)
coverage["years"] = coverage.apply(
    lambda row: f"{row['min'].year}-{row['max'].year}",
    axis=1,
)
print(coverage[["years", "count"]])

                         years   count
Samplingpoint                         
FR/SPO-FR04004_6001  2022-2025   24083
FR/SPO-FR04031_6001  2022-2025   17154
FR/SPO-FR04055_6001  2019-2025   43075
FR/SPO-FR04131_6001  2022-2025   25760
FR/SPO-FR04143_6001  2013-2019   55384
FR/SPO-FR04329_6001  2013-2025  102167


In [14]:
print("Unique Paris PM2.5 stations in the EEA holdings:",
      len(paris_codes))

Unique Paris PM2.5 stations in the EEA holdings: 6


## 5. Notes for the later analysis

This EEA extract contains 534 valid negative readings, with a minimum of -3.0 µg/m³. Retaining these values increases the valid row count from 267,089 to 267,623. This treatment is applied consistently in the later Geod'Air notebooks.

An earlier comparison found that annual means for FR04329 calculated from EEA and Geod'Air differed by about 0.01 µg/m³. That comparison used the previous treatment of negative readings, so I do not use the figure as a formal validation result here. It is not required for the later analysis.

The station-span result is unaffected by retaining negative measurements because it is based on first and last valid timestamps rather than on the concentration mean. FR04143 ends in 2019, FR04055 starts providing PM2.5 records in 2019, three stations enter during 2022, and FR04329 is the only series spanning the full analysis window.

Further exploratory work on comparator coverage, the historical NO₂ series and Great Britain data availability is kept separately in `01b_scouting.ipynb`.